# P-012 · Learning Curve Analysis
**Agent OS v3.0 — Work Packet P-012**

**Question:** Is 1,543 devices enough, or would more data meaningfully improve predictions?

| Field | Value |
|-------|-------|
| Dataset | `perovskite_stability_clean.csv` (1,543 devices) |
| Features | 16 (band gap, composition, process, JV) |
| Target | `log1p(Stability_PCE_T80)` |
| Model | ExtraTreesRegressor (locked config) |
| Current perf | tau-b 0.271 (full), CV 0.289 (5-fold) |
| Panel | Device 850, 1320, 119 |

## 1 — Load Data & Define Features

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.stats import kendalltau
from scipy.optimize import curve_fit

# Load data
df = pd.read_csv("perovskite_stability_clean.csv")
print(f"Dataset: {len(df)} devices")

# Features and target
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'

X = df[FEATURES].values
y = np.log1p(df[TARGET].values)

# Panel device indices
PANEL = [850, 1320, 119]
print(f"Features: {len(FEATURES)}, Target: log1p({TARGET})")
print(f"Panel devices: {PANEL}")

Dataset: 1543 devices
Features: 16, Target: log1p(Stability_PCE_T80)
Panel devices: [850, 1320, 119]


## 2 — Learning Curve: tau-b, MAE, Panel Stability vs Data Fraction

In [2]:

# Fixed test set (20%), seed=42
X_train_full, X_test, y_train_full, y_test, idx_train_full, idx_test = train_test_split(
    X, y, np.arange(len(y)), test_size=0.2, random_state=42
)

print(f"Train size: {len(X_train_full)}, Test size: {len(X_test)}")

# Locked ET config
ET_PARAMS = dict(
    n_estimators=700, max_features='sqrt', min_samples_split=3,
    min_samples_leaf=1, bootstrap=False, random_state=42, n_jobs=-1
)

fractions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
N_SUBSAMPLES = 10
results = []

for frac in fractions:
    n_use = max(1, int(len(X_train_full) * frac))
    tau_list, mae_list = [], []

    for s in range(N_SUBSAMPLES):
        rng = np.random.RandomState(s)
        if frac < 1.0:
            idx_sub = rng.choice(len(X_train_full), size=n_use, replace=False)
        else:
            idx_sub = np.arange(len(X_train_full))

        model = ExtraTreesRegressor(**ET_PARAMS)
        model.fit(X_train_full[idx_sub], y_train_full[idx_sub])
        y_pred = model.predict(X_test)

        tau, _ = kendalltau(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        tau_list.append(tau)
        mae_list.append(mae)

    row = {
        'fraction': frac,
        'n_train': n_use,
        'mean_tau': np.mean(tau_list),
        'std_tau': np.std(tau_list),
        'mean_mae': np.mean(mae_list),
    }
    results.append(row)
    print(f"  frac={frac:.0%}  n={n_use:>5d}  tau={row['mean_tau']:.4f}\u00b1{row['std_tau']:.4f}  "
          f"MAE={row['mean_mae']:.4f}")

lc_df = pd.DataFrame(results)
print("\n--- Learning Curve Summary ---")
print(lc_df.to_string(index=False))

# Panel stability: use multi-split approach (P-005 seeds 0-19) to ensure
# all panel devices get test-set appearances
print("\n--- Panel Stability vs Data Fraction (20 splits, seeds 0-19) ---")
N_PANEL_SPLITS = 20
panel_results = []

for frac in fractions:
    panel_top20 = {d: 0 for d in PANEL}
    panel_appearances = {d: 0 for d in PANEL}
    
    for split_seed in range(N_PANEL_SPLITS):
        Xtr, Xte, ytr, yte, itr, ite = train_test_split(
            X, y, np.arange(len(y)), test_size=0.2, random_state=split_seed
        )
        n_use = max(1, int(len(Xtr) * frac))
        rng = np.random.RandomState(split_seed + 1000)
        if frac < 1.0:
            sub_idx = rng.choice(len(Xtr), size=n_use, replace=False)
        else:
            sub_idx = np.arange(len(Xtr))
        
        model = ExtraTreesRegressor(**ET_PARAMS)
        model.fit(Xtr[sub_idx], ytr[sub_idx])
        preds = model.predict(Xte)
        
        top20_idx = np.argsort(preds)[-20:]
        top20_original = ite[top20_idx]
        
        for d in PANEL:
            if d in ite:
                panel_appearances[d] += 1
                if d in top20_original:
                    panel_top20[d] += 1
    
    prow = {'fraction': frac, 'n_train_approx': n_use}
    for d in PANEL:
        rate = panel_top20[d] / panel_appearances[d] if panel_appearances[d] > 0 else float('nan')
        prow[f'panel_top20_rate_{d}'] = rate
        prow[f'panel_appearances_{d}'] = panel_appearances[d]
    panel_results.append(prow)
    print(f"  frac={frac:.0%}: " + " | ".join(
        f"Dev {d}: {panel_top20[d]}/{panel_appearances[d]}" for d in PANEL
    ))

panel_df = pd.DataFrame(panel_results)
for d in PANEL:
    lc_df[f'panel_top20_rate_{d}'] = panel_df[f'panel_top20_rate_{d}'].values


Train size: 1234, Test size: 309


  frac=10%  n=  123  tau=0.1112±0.0385  MAE=1.4564


  frac=20%  n=  246  tau=0.1572±0.0310  MAE=1.3948


  frac=30%  n=  370  tau=0.1791±0.0170  MAE=1.3698


  frac=40%  n=  493  tau=0.1966±0.0202  MAE=1.3390


...

  frac=60%: Dev 850: 4/4 | Dev 1320: 2/3 | Dev 119: 2/4


  frac=70%: Dev 850: 3/4 | Dev 1320: 3/3 | Dev 119: 2/4


  frac=80%: Dev 850: 3/4 | Dev 1320: 3/3 | Dev 119: 3/4


  frac=90%: Dev 850: 3/4 | Dev 1320: 3/3 | Dev 119: 3/4


  frac=100%: Dev 850: 4/4 | Dev 1320: 3/3 | Dev 119: 4/4


## 3 — Saturation Analysis: Log Curve Fit & Extrapolation

In [3]:
# Fit: tau = a * ln(N) + b
def log_model(n, a, b):
    return a * np.log(n) + b

n_vals = lc_df['n_train'].values.astype(float)
tau_vals = lc_df['mean_tau'].values

popt, pcov = curve_fit(log_model, n_vals, tau_vals)
a, b = popt
print(f"Fitted: tau = {a:.6f} * ln(N) + {b:.6f}")

# Current N (full training set)
N_current = int(n_vals[-1])
tau_current = log_model(N_current, a, b)

# Extrapolate to 2x and 5x
N_2x = 2 * N_current
N_5x = 5 * N_current
tau_2x = log_model(N_2x, a, b)
tau_5x = log_model(N_5x, a, b)

gain_2x = tau_2x - tau_current
gain_5x = tau_5x - tau_current

print(f"\nCurrent N = {N_current}, fitted tau = {tau_current:.4f}")
print(f"  2x data (N={N_2x}): extrapolated tau = {tau_2x:.4f}, gain = +{gain_2x:.4f}")
print(f"  5x data (N={N_5x}): extrapolated tau = {tau_5x:.4f}, gain = +{gain_5x:.4f}")

# Saturation slope: d(tau)/d(N) at current N = a / N
slope_at_current = a / N_current
# Slope per doubling: a * ln(2)
slope_per_doubling = a * np.log(2)
print(f"\nSlope at 100%: d(tau)/d(N) = {slope_at_current:.6f}")
print(f"Tau gain per doubling of data: {slope_per_doubling:.4f}")

Fitted: tau = 0.074183 * ln(N) + -0.253252

Current N = 1234, fitted tau = 0.2748
  2x data (N=2468): extrapolated tau = 0.3262, gain = +0.0514
  5x data (N=6170): extrapolated tau = 0.3942, gain = +0.1194

Slope at 100%: d(tau)/d(N) = 0.000060
Tau gain per doubling of data: 0.0514


## 4 — Save Results

In [4]:
lc_df.to_csv("Packet_P012_Learning_Curve.csv", index=False)
print("Saved: Packet_P012_Learning_Curve.csv")
print(lc_df.to_string(index=False))

Saved: Packet_P012_Learning_Curve.csv
 fraction  n_train  mean_tau  std_tau  mean_mae  panel_top20_rate_850  panel_top20_rate_1320  panel_top20_rate_119
      0.1      123  0.111165 0.038476  1.456392                  0.50               0.000000                  0.00
      0.2      246  0.157238 0.031047  1.394757                  0.50               0.333333                  0.25
      0.3      370  0.179111 0.017034  1.369846                  0.75               1.000000                  0.25
      0.4      493  0.196622 0.020195  1.339004                  1.00               1.000000                  0.50
      0.5      617  0.215180 0.015929  1.309318                  1.00               0.666667                  0.50
      0.6      740  0.236251 0.027074  1.291908                  1.00               0.666667                  0.50
      0.7      863  0.251328 0.019606  1.276555                  0.75               1.000000                  0.50
      0.8      987  0.261225 0.012480  1.2

## 5 — Status Footer

In [5]:

# Honest status determination
SATURATION_THRESHOLD = 0.01  # tau-b per doubling

if abs(slope_per_doubling) < SATURATION_THRESHOLD:
    status = "CONFIRMED"
    verdict = ("The learning curve is clearly saturating. "
               f"Gain per doubling is only {slope_per_doubling:.4f} tau-b (< {SATURATION_THRESHOLD} threshold). "
               f"1,543 devices is sufficient for the current model+features.")
else:
    status = "PROMISING"
    verdict = ("The learning curve is still rising meaningfully. "
               f"Gain per doubling is {slope_per_doubling:.4f} tau-b (>= {SATURATION_THRESHOLD} threshold). "
               f"More data would help.")

print(f"{'=' * 65}")
print(f"P-012 STATUS: {status}")
print(f"{'=' * 65}")
print(verdict)
print(f"\nKey numbers:")
print(f"  Fitted model: tau = {a:.6f} * ln(N) + ({b:.6f})")
print(f"  Tau at N={N_current}: {tau_current:.4f}")
print(f"  Tau at 2x (N={N_2x}): {tau_2x:.4f} (+{gain_2x:.4f})")
print(f"  Tau at 5x (N={N_5x}): {tau_5x:.4f} (+{gain_5x:.4f})")
print(f"  Slope per doubling: {slope_per_doubling:.4f}")

# Panel stability summary
print(f"\nPanel stability vs data fraction (multi-split):")
for d in PANEL:
    col = f'panel_top20_rate_{d}'
    rates = lc_df[col].values
    print(f"  Device {d}: top-20 rate at 100% = {rates[-1]:.0%}, at 50% = {rates[4]:.0%}, at 10% = {rates[0]:.0%}")

print(f"\nWhat this means for partners:")
print(f"  - Current 1,543 devices: tau-b ~{tau_current:.3f}")
print(f"  - Doubling to ~3,000 devices: expected tau-b ~{tau_2x:.3f} (+{gain_2x:.3f})")
print(f"  - The model is NOT saturated — lab partner data will improve predictions")
print(f"  - But gains are logarithmic, not linear — diminishing returns expected")


P-012 STATUS: PROMISING
The learning curve is still rising meaningfully. Gain per doubling is 0.0514 tau-b (>= 0.01 threshold). More data would help.

Key numbers:
  Fitted model: tau = 0.074183 * ln(N) + (-0.253252)
  Tau at N=1234: 0.2748
  Tau at 2x (N=2468): 0.3262 (+0.0514)
  Tau at 5x (N=6170): 0.3942 (+0.1194)
  Slope per doubling: 0.0514

Panel stability vs data fraction (multi-split):
  Device 850: top-20 rate at 100% = 100%, at 50% = 100%, at 10% = 50%
  Device 1320: top-20 rate at 100% = 100%, at 50% = 67%, at 10% = 0%
  Device 119: top-20 rate at 100% = 100%, at 50% = 50%, at 10% = 0%

What this means for partners:
  - Current 1,543 devices: tau-b ~0.275
  - Doubling to ~3,000 devices: expected tau-b ~0.326 (+0.051)
  - The model is NOT saturated — lab partner data will improve predictions
  - But gains are logarithmic, not linear — diminishing returns expected
